# Shown Space Scoring Path Visuals

This notebook uses Shown Space's public game API to recreate field-path visuals from coordinates, then summarizes a team's scoring possessions as an interactive average path and heatmap.

Default team: `glory`.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd

from ufa import (
    average_scoring_path,
    build_scoring_possessions,
    cluster_scoring_possessions,
    create_scoring_possession_browser,
    create_team_scoring_possession_browser,
    create_team_playstyle_report_browser,
    fetch_shownspace_games,
    fetch_shownspace_season_throws,
    fetch_shownspace_throws_for_games,
    plot_average_scoring_path,
    plot_possession_path,
    plot_representative_paths,
    plot_scoring_heatmap,
    plot_team_representative_path_grid,
    select_representative_paths,
    select_top_paths,
    summarize_path_clusters,
    summarize_team_playstyle,
    summarize_team_playstyles,
)

## Settings

Use `MAX_GAMES = 3` first to validate the visual quickly. Set `MAX_GAMES = None` for the full team season.

In [2]:
SEASON = 2026
TEAM_ID = "breeze"
MAX_GAMES = 3
SAMPLE_GAMES_RANDOMLY = True
RANDOM_STATE = 7
UNIQUE_REPRESENTATIVE_GAMES = True
PULL_RECEIVE_SCORES_ONLY = True
LONG_FIELD_ONLY = True
MAX_START_Y = 45
MIN_FIELD_PROGRESS = 50
EXCLUDE_HUCKS_FROM_TOP_PATHS = True

all_games = fetch_shownspace_games(season=SEASON, final_only=True)
team_games = all_games[
    all_games["HomeTeamID"].str.lower().eq(TEAM_ID.lower())
    | all_games["AwayTeamID"].str.lower().eq(TEAM_ID.lower())
].reset_index(drop=True)

if MAX_GAMES is None:
    games = team_games.copy()
elif SAMPLE_GAMES_RANDOMLY:
    games = (
        team_games
        .sample(n=min(MAX_GAMES, len(team_games)), random_state=RANDOM_STATE)
        .sort_values("StartTimestamp")
        .reset_index(drop=True)
    )
else:
    games = team_games.head(MAX_GAMES).copy()

throws = fetch_shownspace_throws_for_games(games["GameID"].tolist(), delay=0.15)

games[["GameID", "AwayTeamID", "HomeTeamID", "AwayScore", "HomeScore", "Status", "StartTimestamp"]]


,GameID,AwayTeamID,HomeTeamID,AwayScore,HomeScore,Status,StartTimestamp
0,2026-05-10-DC-NY,breeze,empire,26,25,Final,2026-05-10 13:00:00
1,2026-06-05-CAR-DC,flyers,breeze,22,24,Final,2026-06-05 19:00:00
2,2026-07-11-PHI-DC,phoenix,breeze,17,28,Final,2026-07-11 19:00:00


In [3]:
possessions, paths = build_scoring_possessions(throws, team_id=TEAM_ID)

print(f"Throws loaded: {len(throws):,}")
print(f"Scoring possessions found for {TEAM_ID}: {len(possessions):,}")

possessions.sort_values("risk_adjusted_aec_per_throw", ascending=False).head(10)

Throws loaded: 1,774
Scoring possessions found for breeze: 78


,possession_id,GameID,team_id,start_timestamp,game_quarter,quarter_point,possession_num,is_home_team,line_type,outcome,...,mean_cp,risk_adjusted_aec_per_throw,total_yards,yards_per_throw,total_throw_distance,avg_throw_distance,max_throw_distance,huck_count,reset_count,lateral_yards
41,2026-06-05-CAR-DC|3|5|2|True,2026-06-05-CAR-DC,breeze,2026-06-05 19:00:00,3,5,2,True,d_line,goal,...,0.954762,0.954762,12.43,12.430,13.616846,13.616846,13.616846,0,0,5.56
25,2026-05-10-DC-NY|5|7|2|False,2026-05-10-DC-NY,breeze,2026-05-10 13:00:00,5,7,2,False,d_line,goal,...,0.950510,0.950510,15.59,15.590,15.725511,15.725511,15.725511,0,0,2.06
46,2026-06-05-CAR-DC|4|3|2|True,2026-06-05-CAR-DC,breeze,2026-06-05 19:00:00,4,3,2,True,d_line,goal,...,0.944195,0.944195,11.60,11.600,19.432141,19.432141,19.432141,0,0,15.59
51,2026-07-11-PHI-DC|1|3|2|True,2026-07-11-PHI-DC,breeze,2026-07-11 19:00:00,1,3,2,True,d_line,goal,...,0.933276,0.933276,17.50,17.500,18.953849,18.953849,18.953849,0,0,7.28
39,2026-06-05-CAR-DC|3|2|2|True,2026-06-05-CAR-DC,breeze,2026-06-05 19:00:00,3,2,2,True,d_line,goal,...,0.825573,0.825573,11.26,11.260,37.233422,37.233422,37.233422,0,0,35.49
28,2026-06-05-CAR-DC|1|7|3|True,2026-06-05-CAR-DC,breeze,2026-06-05 19:00:00,1,7,3,True,o_line,goal,...,0.776833,0.776833,26.49,26.490,43.958236,43.958236,43.958236,1,0,35.08
38,2026-06-05-CAR-DC|3|1|4|True,2026-06-05-CAR-DC,breeze,2026-06-05 19:00:00,3,1,4,True,d_line,goal,...,0.577880,0.577880,57.80,57.800,58.158770,58.158770,58.158770,1,0,6.45
56,2026-07-11-PHI-DC|1|11|4|True,2026-07-11-PHI-DC,breeze,2026-07-11 19:00:00,1,11,4,True,d_line,goal,...,0.544389,0.544389,71.87,71.870,74.649376,74.649376,74.649376,1,0,20.18
72,2026-07-11-PHI-DC|4|2|2|True,2026-07-11-PHI-DC,breeze,2026-07-11 19:00:00,4,2,2,True,d_line,goal,...,0.938994,0.471844,37.21,18.605,37.494460,18.747230,19.361327,0,0,3.92
73,2026-07-11-PHI-DC|4|3|6|True,2026-07-11-PHI-DC,breeze,2026-07-11 19:00:00,4,3,6,True,d_line,goal,...,0.940377,0.469968,31.03,15.515,31.650297,15.825148,20.245960,0,0,5.97


## Long-Field Possession Filter

Use this to focus the visuals on possessions that start farther from the scoring end zone, instead of short-field scores after turnovers.

In [4]:
analysis_possessions = possessions.copy()

if PULL_RECEIVE_SCORES_ONLY:
    analysis_possessions = analysis_possessions[
        analysis_possessions["possession_num"].eq(1)
    ].copy()

if LONG_FIELD_ONLY:
    analysis_possessions = analysis_possessions[
        analysis_possessions["start_y"].le(MAX_START_Y)
        & analysis_possessions["field_progress"].ge(MIN_FIELD_PROGRESS)
    ].copy()

analysis_ids = set(analysis_possessions["possession_id"])
analysis_paths = [
    path for path in paths
    if path["possession_id"].iloc[0] in analysis_ids
]

print(f"All scoring possessions: {len(possessions):,}")
initial_scoring_holds = possessions["possession_num"].eq(1).sum()
print(f"Initial-possession scoring holds: {initial_scoring_holds:,}")
print(f"Analysis possessions: {len(analysis_possessions):,}")

analysis_possessions[[
    "possession_id", "GameID", "possession_num", "start_y", "end_y",
    "field_progress", "throw_count", "total_aec", "aec_per_throw"
]].head(10)


All scoring possessions: 78
Initial-possession scoring holds: 41
Analysis possessions: 39


,possession_id,GameID,possession_num,start_y,end_y,field_progress,throw_count,total_aec,aec_per_throw
0,2026-05-10-DC-NY|1|1|1|False,2026-05-10-DC-NY,1,11.87,107.16,95.29,9,1.089516,0.121057
2,2026-05-10-DC-NY|1|5|1|False,2026-05-10-DC-NY,1,8.59,114.24,105.65,11,1.030926,0.093721
3,2026-05-10-DC-NY|1|7|1|False,2026-05-10-DC-NY,1,4.29,111.07,106.78,15,1.107679,0.073845
5,2026-05-10-DC-NY|2|7|1|False,2026-05-10-DC-NY,1,11.94,112.37,100.43,4,1.001092,0.250273
7,2026-05-10-DC-NY|2|11|1|False,2026-05-10-DC-NY,1,10.91,106.38,95.47,11,1.015889,0.092354
8,2026-05-10-DC-NY|2|13|1|False,2026-05-10-DC-NY,1,11.25,110.32,99.07,6,1.070848,0.178475
10,2026-05-10-DC-NY|3|2|1|False,2026-05-10-DC-NY,1,40.00,106.95,66.95,6,1.048643,0.174774
11,2026-05-10-DC-NY|3|4|1|False,2026-05-10-DC-NY,1,7.89,105.58,97.69,10,0.986872,0.098687
13,2026-05-10-DC-NY|3|8|1|False,2026-05-10-DC-NY,1,1.02,112.58,111.56,7,1.000883,0.142983
18,2026-05-10-DC-NY|4|3|1|False,2026-05-10-DC-NY,1,2.26,106.27,104.01,4,0.721058,0.180264


## Glory Scoring Possession Browser

This is the Shown Space-style possession browser. It uses a custom HTML/SVG field instead of Plotly so the field has simple lines, dots, hover tooltips, and no chart toolbar.

In [5]:
BROWSER_SEASON = 2026
BROWSER_TEAM_ID = "glory"
BROWSER_MAX_GAMES = None  # None means every final game for the team

# Defaults to all Glory scoring possessions. Turn these on only if you want a narrower view.
BROWSER_PULL_RECEIVE_SCORES_ONLY = False
BROWSER_LONG_FIELD_ONLY = False
BROWSER_MAX_START_Y = 45
BROWSER_MIN_FIELD_PROGRESS = 50
BROWSER_EXCLUDE_HUCKS = False

browser_all_games = fetch_shownspace_games(season=BROWSER_SEASON, final_only=True)
browser_games = browser_all_games[
    browser_all_games["HomeTeamID"].str.lower().eq(BROWSER_TEAM_ID.lower())
    | browser_all_games["AwayTeamID"].str.lower().eq(BROWSER_TEAM_ID.lower())
].sort_values("StartTimestamp").reset_index(drop=True)

if BROWSER_MAX_GAMES is not None:
    browser_games = browser_games.head(BROWSER_MAX_GAMES).copy()

browser_throws = fetch_shownspace_throws_for_games(
    browser_games["GameID"].tolist(),
    delay=0.15,
)
browser_possessions, browser_paths = build_scoring_possessions(
    browser_throws,
    team_id=BROWSER_TEAM_ID,
)

if BROWSER_PULL_RECEIVE_SCORES_ONLY:
    browser_possessions = browser_possessions[
        browser_possessions["possession_num"].eq(1)
    ].copy()

if BROWSER_LONG_FIELD_ONLY:
    browser_possessions = browser_possessions[
        browser_possessions["start_y"].le(BROWSER_MAX_START_Y)
        & browser_possessions["field_progress"].ge(BROWSER_MIN_FIELD_PROGRESS)
    ].copy()

if BROWSER_EXCLUDE_HUCKS:
    browser_possessions = browser_possessions[
        browser_possessions["huck_count"].fillna(0).eq(0)
    ].copy()

browser_ids = set(browser_possessions["possession_id"])
browser_paths = [
    path for path in browser_paths
    if path["possession_id"].iloc[0] in browser_ids
]

print(f"Games loaded: {len(browser_games):,}")
print(f"Throws loaded: {len(browser_throws):,}")
print(f"Browser possessions: {len(browser_possessions):,}")

browser_possessions[[
    "possession_id", "GameID", "game_quarter", "quarter_point",
    "possession_num", "throw_count", "start_y", "end_y",
    "field_progress", "total_aec", "aec_per_throw"
]].head(10)


Games loaded: 12
Throws loaded: 6,588
Browser possessions: 261


,possession_id,GameID,game_quarter,quarter_point,possession_num,throw_count,start_y,end_y,field_progress,total_aec,aec_per_throw
0,2026-04-25-DC-BOS|1|1|2|True,2026-04-25-DC-BOS,1,1,2,2,75.03,102.12,27.09,1.010065,0.505033
1,2026-04-25-DC-BOS|1|3|1|True,2026-04-25-DC-BOS,1,3,1,10,14.77,106.96,92.19,0.987499,0.098750
2,2026-04-25-DC-BOS|1|4|2|True,2026-04-25-DC-BOS,1,4,2,10,34.12,115.29,81.17,1.145452,0.114545
3,2026-04-25-DC-BOS|1|5|2|True,2026-04-25-DC-BOS,1,5,2,20,19.74,104.06,84.32,0.891622,0.044581
4,2026-04-25-DC-BOS|1|6|2|True,2026-04-25-DC-BOS,1,6,2,7,49.80,108.32,58.52,1.004663,0.143523
5,2026-04-25-DC-BOS|1|8|1|True,2026-04-25-DC-BOS,1,8,1,9,46.70,115.16,68.46,1.000726,0.111192
6,2026-04-25-DC-BOS|2|1|1|True,2026-04-25-DC-BOS,2,1,1,15,8.45,105.29,96.84,1.041824,0.069455
7,2026-04-25-DC-BOS|2|2|2|True,2026-04-25-DC-BOS,2,2,2,3,89.80,105.61,15.81,1.006915,0.335638
8,2026-04-25-DC-BOS|2|4|1|True,2026-04-25-DC-BOS,2,4,1,10,13.80,114.32,100.52,0.984443,0.098444
9,2026-04-25-DC-BOS|2|6|1|True,2026-04-25-DC-BOS,2,6,1,4,13.87,107.54,93.67,1.000987,0.250247


## Widget Display Check

Run this small check if the browser output looks blank. If the slider does not appear, restart the kernel and rerun the import cell.

In [6]:
import ipywidgets as widgets
widgets.IntSlider(description="widget test")


IntSlider(value=0, description='widget test')

In [7]:
import sys
sys.path.insert(0, "../src")

import importlib
import ufa.shownspace_paths as shownspace_paths
importlib.reload(shownspace_paths)

create_scoring_possession_browser = shownspace_paths.create_scoring_possession_browser
create_team_scoring_possession_browser = shownspace_paths.create_team_scoring_possession_browser

In [9]:
from IPython.display import display

team_browser = create_team_scoring_possession_browser(
    season=BROWSER_SEASON,
    default_team_id=BROWSER_TEAM_ID,
    final_only=True,
    max_games=BROWSER_MAX_GAMES,
    pull_receive_scores_only=BROWSER_PULL_RECEIVE_SCORES_ONLY,
    long_field_only=BROWSER_LONG_FIELD_ONLY,
    max_start_y=BROWSER_MAX_START_Y,
    min_field_progress=BROWSER_MIN_FIELD_PROGRESS,
    exclude_hucks=BROWSER_EXCLUDE_HUCKS,
    n_shape_clusters=8,
)

display(team_browser)


## Team Playstyle Summary Report

These summaries use the same scoring-possession shape features as the browser. The text is rule-based, so every phrase traces back to the metrics shown in the table.


In [ ]:
PLAYSTYLE_COLUMNS = [
    "team_id",
    "possessions",
    "primary_shapes",
    "attack_spaces",
    "pace_style",
    "field_width_style",
    "huck_usage",
    "reset_usage",
    "efficiency_note",
    "playstyle_summary",
]

BACKING_METRIC_COLUMNS = [
    "avg_throws",
    "avg_width",
    "avg_side_switches",
    "avg_directness",
    "avg_middle_usage",
    "avg_sideline_usage",
    "avg_hucks",
    "avg_resets",
    "avg_aec_per_throw",
    "avg_cp",
]

team_playstyle = summarize_team_playstyle(
    browser_possessions,
    browser_paths,
    team_id=BROWSER_TEAM_ID,
    n_shape_clusters=8,
)

team_playstyle_table = pd.DataFrame([team_playstyle])

display(
    create_team_playstyle_report_browser(
        team_playstyle,
        team_playstyle_table,
        title=f"{BROWSER_TEAM_ID.title()} playstyle summary, {BROWSER_SEASON}",
    )
)


HTML(value='\n    <div class="ufa-playstyle-browser">\n      <style>\n        .ufa-playstyle-browser {\n      …

In [ ]:
PLAYSTYLE_TEAM_IDS = ["glory", "empire", "spiders"]
PLAYSTYLE_MAX_GAMES = None

playstyle_rows = []
for playstyle_team_id in PLAYSTYLE_TEAM_IDS:
    team_games = browser_all_games[
        browser_all_games["HomeTeamID"].str.lower().eq(playstyle_team_id.lower())
        | browser_all_games["AwayTeamID"].str.lower().eq(playstyle_team_id.lower())
    ].sort_values("StartTimestamp").reset_index(drop=True)

    if PLAYSTYLE_MAX_GAMES is not None:
        team_games = team_games.head(PLAYSTYLE_MAX_GAMES).copy()

    team_throws = fetch_shownspace_throws_for_games(
        team_games["GameID"].tolist(),
        delay=0.15,
    )
    team_possessions, team_paths = build_scoring_possessions(
        team_throws,
        team_id=playstyle_team_id,
    )
    playstyle_rows.append(
        summarize_team_playstyle(
            team_possessions,
            team_paths,
            team_id=playstyle_team_id,
            n_shape_clusters=8,
        )
    )

team_playstyle_table = pd.DataFrame(playstyle_rows)
selected_team_row = team_playstyle_table[
    team_playstyle_table["team_id"].astype(str).str.lower().eq(BROWSER_TEAM_ID.lower())
]
if selected_team_row.empty:
    selected_team_playstyle = team_playstyle_table.iloc[0]
else:
    selected_team_playstyle = selected_team_row.iloc[0]

display(
    create_team_playstyle_report_browser(
        selected_team_playstyle,
        team_playstyle_table,
        title=f"Team playstyle comparison, {BROWSER_SEASON}",
    )
)


HTML(value='\n    <div class="ufa-playstyle-browser">\n      <style>\n        .ufa-playstyle-browser {\n      …

## Average Scoring Path

The average path is progress-normalized. Each scoring possession is resampled to fixed progress checkpoints from possession start to goal, then the checkpoint coordinates are averaged.

In [ ]:
avg_path = average_scoring_path(paths)
avg_path

,checkpoint,x,y,mean_cumulative_aec,mean_cp,mean_win_prob,possessions
0,0.0,-0.745965,33.526667,0.002780,0.957209,0.463772,57
1,0.2,-1.633534,48.079409,0.146931,0.957895,0.463052,57
2,0.4,-1.526186,62.116529,0.302248,0.935293,0.461256,57
3,0.6,-0.669802,76.793543,0.492257,0.897338,0.459412,57
4,0.8,0.908583,91.471002,0.706195,0.873419,0.457360,57
5,1.0,2.396316,106.974912,0.993791,0.850729,0.455316,57


In [ ]:
fig = plot_average_scoring_path(
    avg_path,
    paths=paths,
    title=f"{TEAM_ID.title()} average scoring path, {SEASON} sample",
    show_individual_paths=True,
)
fig.show()

## Real Representative Paths

The mean path is useful as a center-of-gravity check, but it can hide the actual bends, resets, hucks, and lateral movement that make an offense interesting. These cells keep real possessions intact and then pick examples worth studying.

In [ ]:
clustered_possessions = cluster_scoring_possessions(analysis_possessions, analysis_paths, n_clusters=4)
cluster_summary = summarize_path_clusters(clustered_possessions)
cluster_summary

,path_cluster,style,possessions,avg_throws,avg_aec_per_throw,avg_cp,avg_yards_per_throw,avg_max_throw_distance,avg_resets,avg_lateral_yards,avg_width,avg_directness,avg_side_switches,avg_middle_third_share,avg_sideline_share,avg_red_zone_entry_x
0,0,huck score,9,6.0,0.184375,0.902844,17.471060,60.148577,1.222222,61.942222,30.365556,0.705808,1.555556,0.447443,0.188316,6.150000
3,1,methodical,5,10.8,0.095580,0.962772,8.635326,28.078374,1.800000,92.324000,35.226000,0.590571,3.200000,0.419231,0.135577,-8.262000
5,1,reset-heavy,4,12.5,0.082973,0.954716,7.247516,35.367249,4.000000,148.732500,43.482500,0.437074,4.750000,0.382372,0.306410,-8.482500
2,1,huck score,3,8.0,0.129148,0.909161,13.427528,52.153731,1.666667,97.296667,39.126667,0.595740,3.000000,0.384722,0.280556,-18.393333
6,2,huck score,2,3.0,0.336697,0.864603,25.576667,60.391228,0.000000,26.520000,19.235000,0.913077,1.500000,0.833333,0.000000,-7.075000
4,1,mixed,2,5.5,0.184203,0.932560,11.998393,33.485890,1.000000,71.960000,36.000000,0.668279,2.000000,0.428571,0.187500,-6.595000
9,3,methodical,2,10.0,0.110901,0.937435,8.391458,30.871799,1.000000,129.695000,40.735000,0.473464,4.000000,0.354167,0.437500,14.310000
10,3,reset-heavy,2,17.0,0.060715,0.957203,4.582857,33.459627,5.500000,224.815000,38.305000,0.268142,8.500000,0.451786,0.200000,13.310000
8,2,mixed,1,5.0,0.200129,0.933920,17.314000,37.564298,0.000000,47.640000,18.600000,0.846000,2.000000,0.500000,0.000000,-13.360000
1,0,methodical,1,10.0,0.094519,0.961842,8.227000,29.920623,2.000000,106.600000,31.780000,0.561850,4.000000,0.400000,0.000000,16.570000


In [ ]:
# Try to show each representative path from a different game when possible.
representative_paths = select_representative_paths(
    clustered_possessions,
    analysis_paths,
    group_column="path_cluster",
    unique_games=UNIQUE_REPRESENTATIVE_GAMES,
)

rep_fig = plot_representative_paths(
    representative_paths,
    title=f"{TEAM_ID.title()} representative scoring path styles, {SEASON} sample",
)
rep_fig.show()

In [ ]:
top_path_possessions = clustered_possessions.copy()
top_path_source_paths = analysis_paths

if EXCLUDE_HUCKS_FROM_TOP_PATHS:
    top_path_possessions = top_path_possessions[
        top_path_possessions["huck_count"].fillna(0).eq(0)
    ].copy()
    top_path_ids = set(top_path_possessions["possession_id"])
    top_path_source_paths = [
        path for path in analysis_paths
        if path["possession_id"].iloc[0] in top_path_ids
    ]

top_paths = select_top_paths(
    top_path_possessions,
    top_path_source_paths,
    metric="aec_per_throw",
    n=3,
)

top_path_title = "highest non-huck long-field aEC per throw" if EXCLUDE_HUCKS_FROM_TOP_PATHS else "highest long-field aEC per throw"

best_fig = plot_possession_path(
    top_paths[0],
    title=f"{TEAM_ID.title()} {top_path_title} scoring possession, {SEASON} sample",
)
best_fig.show()


## Middle Non-Huck Scoring Possessions

The highest `aEC_per_throw` possession can still be an outlier. This view sorts the filtered non-huck possessions by `aEC_per_throw` and plots the middle five, which should be closer to normal efficient offense.

In [ ]:
MIDDLE_PATH_COUNT = 5
MIDDLE_PATH_METRIC = "aec_per_throw"

middle_source = top_path_possessions.sort_values(MIDDLE_PATH_METRIC).reset_index(drop=True)
middle_count = min(MIDDLE_PATH_COUNT, len(middle_source))
middle_start = max((len(middle_source) - middle_count) // 2, 0)
middle_path_possessions = middle_source.iloc[
    middle_start:middle_start + middle_count
].copy()

middle_path_lookup = {
    path["possession_id"].iloc[0]: path
    for path in top_path_source_paths
}
middle_paths = {
    f"middle {rank + 1}: {row[MIDDLE_PATH_METRIC]:.3f}": middle_path_lookup[row["possession_id"]]
    for rank, (_, row) in enumerate(middle_path_possessions.iterrows())
    if row["possession_id"] in middle_path_lookup
}

middle_path_possessions[[
    "possession_id", "GameID", "start_y", "end_y", "field_progress",
    "throw_count", "huck_count", MIDDLE_PATH_METRIC
]]


,possession_id,GameID,start_y,end_y,field_progress,throw_count,huck_count,aec_per_throw
6,2026-04-25-DC-BOS|2|7|1|False,2026-04-25-DC-BOS,5.62,102.21,96.59,12,0,0.083578
7,2026-05-29-DC-TOR|1|9|1|False,2026-05-29-DC-TOR,17.11,105.43,88.32,13,0,0.083842
8,2026-04-25-DC-BOS|2|5|1|False,2026-04-25-DC-BOS,13.59,104.41,90.82,10,0,0.089218
9,2026-04-25-DC-BOS|4|4|1|False,2026-04-25-DC-BOS,17.02,108.67,91.65,10,0,0.094519
10,2026-06-13-BOS-DC|4|12|1|True,2026-06-13-BOS-DC,17.29,105.30,88.01,12,0,0.098522


In [ ]:
middle_fig = plot_representative_paths(
    middle_paths,
    title=f"{TEAM_ID.title()} middle {len(middle_paths)} non-huck long-field scoring possessions, {SEASON} sample",
)
middle_fig.show()


## Compare Teams Side By Side

The readable comparison is one possession style at a time. Build representative paths for each team, then choose a style like `huck`, `reset`, `quick`, or `methodical` to compare across teams.

In [ ]:
TEAM_IDS_TO_COMPARE = ["glory", "empire", "spiders"]

team_representative_paths = {}
team_cluster_summaries = {}

for compare_team_id in TEAM_IDS_TO_COMPARE:
    compare_games = all_games[
        all_games["HomeTeamID"].str.lower().eq(compare_team_id.lower())
        | all_games["AwayTeamID"].str.lower().eq(compare_team_id.lower())
    ].reset_index(drop=True)

    if MAX_GAMES is None:
        selected_games = compare_games.copy()
    elif SAMPLE_GAMES_RANDOMLY:
        selected_games = (
            compare_games
            .sample(n=min(MAX_GAMES, len(compare_games)), random_state=RANDOM_STATE)
            .sort_values("StartTimestamp")
            .reset_index(drop=True)
        )
    else:
        selected_games = compare_games.head(MAX_GAMES).copy()

    compare_throws = fetch_shownspace_throws_for_games(
        selected_games["GameID"].tolist(),
        delay=0.15,
    )
    compare_possessions, compare_paths = build_scoring_possessions(
        compare_throws,
        team_id=compare_team_id,
    )

    compare_analysis_possessions = compare_possessions.copy()
    if PULL_RECEIVE_SCORES_ONLY:
        compare_analysis_possessions = compare_analysis_possessions[
            compare_analysis_possessions["possession_num"].eq(1)
        ].copy()
    if LONG_FIELD_ONLY:
        compare_analysis_possessions = compare_analysis_possessions[
            compare_analysis_possessions["start_y"].le(MAX_START_Y)
            & compare_analysis_possessions["field_progress"].ge(MIN_FIELD_PROGRESS)
        ].copy()

    compare_ids = set(compare_analysis_possessions["possession_id"])
    compare_analysis_paths = [
        path for path in compare_paths
        if path["possession_id"].iloc[0] in compare_ids
    ]

    compare_clustered = cluster_scoring_possessions(
        compare_analysis_possessions,
        compare_analysis_paths,
        n_clusters=4,
    )
    team_cluster_summaries[compare_team_id] = summarize_path_clusters(compare_clustered)
    team_representative_paths[compare_team_id] = select_representative_paths(
        compare_clustered,
        compare_analysis_paths,
        group_column="path_cluster",
        unique_games=UNIQUE_REPRESENTATIVE_GAMES,
    )

comparison_counts = pd.DataFrame([
    {
        "team_id": team_id,
        "representative_paths": len(representative_paths),
    }
    for team_id, representative_paths in team_representative_paths.items()
])
comparison_counts


,team_id,representative_paths
0,glory,4
1,empire,4
2,spiders,4


In [ ]:
STYLE_TO_COMPARE = "huck"  # Try "reset", "quick", "methodical", or None

style_title = (
    "All styles"
    if STYLE_TO_COMPARE is None
    else STYLE_TO_COMPARE.title()
)

comparison_fig = plot_team_representative_path_grid(
    team_representative_paths,
    title=f"{style_title} representative scoring paths by team, {SEASON}",
    style_filter=STYLE_TO_COMPARE,
    show_arrows=False,
)
comparison_fig.show()


### Optional: All Styles

This is busier, but useful as a quick overview after the one-style comparison makes sense.

In [ ]:
all_styles_fig = plot_team_representative_path_grid(
    team_representative_paths,
    title=f"All representative scoring path styles by team, {SEASON}",
    style_filter=None,
    show_arrows=False,
)
all_styles_fig.show()


## Catch Location Heatmap

This shows where completed throws in scoring possessions are caught.

In [ ]:
heatmap = plot_scoring_heatmap(
    analysis_paths,
    title=f"{TEAM_ID.title()} scoring-possession catch heatmap, {SEASON} sample",
)
heatmap.show()

## Full Team Season

After the sample plots look right, run the full team season by setting `MAX_GAMES = None` below.

In [ ]:
MAX_GAMES = None

games_full, throws_full = fetch_shownspace_season_throws(
    season=SEASON,
    team_id=TEAM_ID,
    max_games=MAX_GAMES,
    delay=0.15,
)

possessions_full, paths_full = build_scoring_possessions(throws_full, team_id=TEAM_ID)
avg_path_full = average_scoring_path(paths_full)

print(f"Games loaded: {len(games_full):,}")
print(f"Throws loaded: {len(throws_full):,}")
print(f"Scoring possessions found for {TEAM_ID}: {len(possessions_full):,}")

possessions_full.sort_values("risk_adjusted_aec_per_throw", ascending=False).head(20)

Games loaded: 10
Throws loaded: 5,671
Scoring possessions found for spiders: 257


,possession_id,GameID,team_id,start_timestamp,game_quarter,quarter_point,possession_num,is_home_team,line_type,start_x,...,mean_cp,risk_adjusted_aec_per_throw,total_yards,yards_per_throw,total_throw_distance,avg_throw_distance,max_throw_distance,huck_count,reset_count,lateral_yards
22,2026-04-26-ORE-OAK|3|8|2|True,2026-04-26-ORE-OAK,spiders,2026-04-26 15:30:00,3,8,2,True,d_line,-8.37,...,0.976773,0.976773,3.96,3.960,4.609902,4.609902,4.609902,0,0,2.36
42,2026-05-02-SLC-OAK|2|6|2|True,2026-05-02-SLC-OAK,spiders,2026-05-02 18:00:00,2,6,2,True,d_line,-18.62,...,0.973911,0.973911,3.77,3.770,5.083404,5.083404,5.083404,0,0,3.41
11,2026-04-26-ORE-OAK|2|3|2|True,2026-04-26-ORE-OAK,spiders,2026-04-26 15:30:00,2,3,2,True,d_line,6.19,...,0.972435,0.972435,6.91,6.910,6.916083,6.916083,6.916083,0,0,0.29
240,2026-06-27-OAK-SLC|2|8|2|False,2026-06-27-OAK-SLC,spiders,2026-06-27 19:00:00,2,8,2,False,d_line,6.86,...,0.964945,0.964945,8.52,8.520,9.858600,9.858600,9.858600,0,0,4.96
79,2026-05-09-OAK-SEA|4|2|2|False,2026-05-09-OAK-SEA,spiders,2026-05-09 15:00:00,4,2,2,False,d_line,13.36,...,0.953415,0.953415,7.65,7.650,12.181662,12.181662,12.181662,0,0,9.48
117,2026-05-17-SD-OAK|2|1|3|True,2026-05-17-SD-OAK,spiders,2026-05-17 15:00:00,2,1,3,True,o_line,-0.60,...,0.944834,0.944834,16.55,16.550,16.658706,16.658706,16.658706,0,0,1.90
113,2026-05-17-SD-OAK|1|4|3|True,2026-05-17-SD-OAK,spiders,2026-05-17 15:00:00,1,4,3,True,o_line,-15.95,...,0.940206,0.940206,17.82,17.820,18.661297,18.661297,18.661297,0,0,5.54
208,2026-06-26-OAK-COL|1|1|1|False,2026-06-26-OAK-COL,spiders,2026-06-26 19:00:00,1,1,1,False,o_line,-2.41,...,0.939296,0.939296,12.36,12.360,16.006202,16.006202,16.006202,0,0,10.17
74,2026-05-09-OAK-SEA|3|8|2|False,2026-05-09-OAK-SEA,spiders,2026-05-09 15:00:00,3,8,2,False,d_line,-18.96,...,0.928043,0.928043,20.53,20.530,20.539060,20.539060,20.539060,0,0,0.61
104,2026-05-10-OAK-ORE|3|7|2|False,2026-05-10-OAK-ORE,spiders,2026-05-10 14:00:00,3,7,2,False,d_line,-10.55,...,0.893104,0.893104,25.61,25.610,26.050432,26.050432,26.050432,0,0,4.77


In [ ]:
fig_full = plot_average_scoring_path(
    avg_path_full,
    paths=paths_full,
    title=f"{TEAM_ID.title()} average scoring path, {SEASON}",
    show_individual_paths=True,
)
fig_full.show()

In [ ]:
heatmap_full = plot_scoring_heatmap(
    paths_full,
    title=f"{TEAM_ID.title()} scoring-possession catch heatmap, {SEASON}",
)
heatmap_full.show()